In [ ]:
import pandas as pd
from pathlib import Path

#Variables
PICKUP, DROPOFF = "tpep_pickup_datetime", "tpep_dropoff_datetime"
COLUMNS = [PICKUP, DROPOFF, "trip_distance", "PULocationID", "DOLocationID"]
DISTANCE_THRESHOLD = 40
INVALID_ZONE_START = 264  # zones 264 and 265 are unknown/outside NYC

def read_columns(path):
    return pd.read_parquet(path, columns=COLUMNS)

# returns string for what data was and is currently and what ammount was dropped
def report(before, after):
    return f"Previous Dataset Size: {before:,} \nCurrent Dataset Size: {after:,} \nDropped: {before - after:,} lines \n{1-(after / before):.2%} Dropped \n{after / before:.2%} Preserved"

# applies every cleaning step in order, prints a report per step + overall, returns cleaned df
def clean_data(df, verbose=True):
    start = len(df)
    df = df.copy()
    df["duration_min"] = (df[DROPOFF] - df[PICKUP]).dt.total_seconds() / 60
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
    df["pickup_dow"] = df["tpep_pickup_datetime"].dt.dayofweek

    steps = [
        ("Duration > 0 & < 120", lambda d: (d["duration_min"] > 0) & (d["duration_min"] < 120)),
        (f"Distance in (0, {DISTANCE_THRESHOLD})]", lambda d: d["trip_distance"].between(0, DISTANCE_THRESHOLD, inclusive="right")),
        ("Drop zones 264/265", lambda d: (d["PULocationID"] < INVALID_ZONE_START) & (d["DOLocationID"] < INVALID_ZONE_START)),
    ]

    # selects a step/cleaning method and apply it to the df and reports the changes
    for name, mask in steps:
        before = len(df)
        df = df.loc[mask(df)].reset_index(drop=True)
        if verbose:
            print(f"--- {name} ---\n{report(before, len(df))}\n")

    if verbose:
        print(f"=== Overall ===\n{report(start, len(df))}")
    return df

In [19]:
df = read_columns("OriginalData_DoNotDelete/yellow_tripdata_2026-Jan.parquet")
df.shape[0]

3724889

In [20]:
df_clean = clean_data(df)

--- Duration > 0 & < 120 ---
Previous Dataset Size: 3,724,889 
Current Dataset Size: 3,675,292 
Dropped: 49,597 lines 
1.33% Dropped 
98.67% Preserved

--- Distance in (0, 40)] ---
Previous Dataset Size: 3,675,292 
Current Dataset Size: 3,549,755 
Dropped: 125,537 lines 
3.42% Dropped 
96.58% Preserved

--- Drop zones 264/265 ---
Previous Dataset Size: 3,549,755 
Current Dataset Size: 3,527,348 
Dropped: 22,407 lines 
0.63% Dropped 
99.37% Preserved

=== Overall ===
Previous Dataset Size: 3,724,889 
Current Dataset Size: 3,527,348 
Dropped: 197,541 lines 
5.30% Dropped 
94.70% Preserved


In [21]:
#save to new parquet file
path_to_data = Path("CleanedData/yellow_tripdata_2026-Jan_Cleaned-Dur-Dis-Zone264-265.parquet")
path_to_data.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(path_to_data)
print(f"Saved to {path_to_data}")

Saved to CleanedData\yellow_tripdata_2026-Jan_Cleaned-Dur-Dis-Zone264-265.parquet


In [ ]:
# Testing 
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error

df_clean = pd.read_parquet(path_to_data)

FEATURES = [
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "pickup_hour",
    "pickup_dow",
]

numeric = ["trip_distance"]
categorical = ["PULocationID", "DOLocationID", "pickup_hour", "pickup_dow"]

X = df_clean[FEATURES]
y = df_clean["duration_min"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# sorts features into numerical or categorical data since some data aren't numerical e.g zone id 250 isn't techically bigger than zone id 5
preprocess = ColumnTransformer([
    ("num", "passthrough", numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])

model = Pipeline([
    ("prep", preprocess),
    ("reg", LinearRegression()),
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# the typical size of prediction errors, with larger errors weighted more heavily
rmse = root_mean_squared_error(y_test, y_pred)

# the proportion of the variation in trip duration that the model explains
r2 = r2_score(y_test, y_pred)

print(f"Root Mean Squared Error: {rmse:.2f} minutes")
print(f"R-squared ($R^2$): {r2:.2f}")

Root Mean Squared Error: 7.65 minutes
R-squared ($R^2$): 0.67
